# SAP ECC Treasury Data Generator
**Purpose:** Generate a realistic 10,000-row SAP ECC treasury export for a mixed IT distribution company (hardware, software, services).  
**Output:** Raw CSV and Excel files simulating a daily treasury analyst extract with intentional data quality issues.

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import string
import os

np.random.seed(42)
random.seed(42)

NUM_ROWS = 10000

# Output directory — change this to wherever you want the files
OUTPUT_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Output directory: {OUTPUT_DIR}')
print('Setup complete.')

Output directory: c:\Users\bobur\Downloads
Setup complete.


## 2. Master Data Dictionaries
These define the business universe: legal entities, banks, vendors, customers, and transaction classifications.  
Every value is chosen to reflect a realistic German-headquartered IT distributor operating across DACH (Germany, Austria, Switzerland).

In [2]:
# === COMPANY CODES (Legal entities in the group) ===
company_codes = {
    '1000': 'TechDist Holding GmbH',
    '2000': 'TechDist Deutschland GmbH',
    '3000': 'TechDist Austria GmbH',
    '4000': 'TechDist Schweiz AG'
}

# === HOUSE BANKS (Where the company holds accounts) ===
house_banks = {
    'DB01': {'name': 'Deutsche Bank', 'accounts': ['HKTID_01', 'HKTID_02']},
    'CB01': {'name': 'Commerzbank', 'accounts': ['HKTID_01']},
    'UB01': {'name': 'UBS', 'accounts': ['HKTID_01']},
    'RB01': {'name': 'Raiffeisen Bank', 'accounts': ['HKTID_01']}
}

# === VENDORS (OEMs and suppliers — who treasury pays) ===
# Note: US vendors are USD-denominated, creating the FX exposure treasury manages
vendors = {
    'V001': {'name': 'Cisco Systems International', 'currency': 'USD'},
    'V002': {'name': 'Dell Technologies GmbH', 'currency': 'EUR'},
    'V003': {'name': 'Hewlett Packard Enterprise', 'currency': 'USD'},
    'V004': {'name': 'Microsoft Ireland Operations', 'currency': 'EUR'},
    'V005': {'name': 'Lenovo Global Technology', 'currency': 'USD'},
    'V006': {'name': 'VMware International Unlimited', 'currency': 'EUR'},
    'V007': {'name': 'Palo Alto Networks', 'currency': 'USD'},
    'V008': {'name': 'SAP SE', 'currency': 'EUR'},
    'V009': {'name': 'Juniper Networks', 'currency': 'USD'},
    'V010': {'name': 'Fortinet Inc.', 'currency': 'USD'}
}

# === CUSTOMERS (Resellers and enterprises — who pays us) ===
# Mostly EUR except Swiss customer in CHF
customers = {
    'C001': {'name': 'Bechtle AG', 'currency': 'EUR'},
    'C002': {'name': 'CANCOM SE', 'currency': 'EUR'},
    'C003': {'name': 'SoftwareOne Deutschland', 'currency': 'EUR'},
    'C004': {'name': 'Computacenter AG', 'currency': 'EUR'},
    'C005': {'name': 'Axians Deutschland', 'currency': 'EUR'},
    'C006': {'name': 'T-Systems International', 'currency': 'EUR'},
    'C007': {'name': 'Atos IT Solutions', 'currency': 'EUR'},
    'C008': {'name': 'Swisscom IT Services', 'currency': 'CHF'}
}

# === PROFIT CENTERS (Maps to business divisions) ===
profit_centers = {
    'PC_HW': 'Hardware Distribution',
    'PC_SW': 'Software & Licensing',
    'PC_SV': 'Professional Services',
    'PC_MS': 'Managed Services'
}

# === DOCUMENT TYPES (SAP standard codes) — weighted by frequency ===
doc_types = {
    'KZ': 0.30,   # Vendor payment (biggest chunk for a distributor)
    'DZ': 0.25,   # Customer incoming payment
    'SA': 0.20,   # G/L account posting (intercompany, manual)
    'ZP': 0.15,   # Payment run output
    'AB': 0.10    # Accounting document (clearing, reversals)
}

# === DEAL TYPES ===
deal_types = {
    'FX_SPOT': 0.25,
    'FX_FORWARD': 0.15,
    'PAYMENT': 0.35,
    'TRANSFER': 0.15,
    'DEPOSIT': 0.10
}

# === PAYMENT METHODS — SEPA dominates in Europe ===
payment_methods = {
    'T': 0.45,   # Bank Transfer (SEPA)
    'W': 0.25,   # Wire (SWIFT) — for USD vendor payments
    'C': 0.05,   # Check — rare but still exists
    'E': 0.25    # Electronic/ACH
}

# === CURRENCIES with base EUR conversion rates ===
currencies_with_rates = {
    'EUR': 1.0,
    'USD': 0.92,
    'CHF': 1.04,
    'GBP': 1.17,
    'JPY': 0.0061
}

# === G/L ACCOUNTS (SAP treasury-relevant, SKR03/04 structure) ===
gl_accounts = {
    '1100000': 'Deutsche Bank EUR Main',
    '1100001': 'Deutsche Bank USD',
    '1100010': 'Commerzbank EUR',
    '1100020': 'UBS CHF',
    '1100030': 'Raiffeisen EUR',
    '1600000': 'Vendor Clearing - Domestic',
    '1600001': 'Vendor Clearing - Foreign',
    '1400000': 'Customer Clearing - Domestic',
    '1400001': 'Customer Clearing - Foreign',
    '4800000': 'Realized FX Gains',
    '4800001': 'Realized FX Losses',
    '4800010': 'Unrealized FX Gains',
    '4800011': 'Unrealized FX Losses',
    '1130000': 'Short-term Deposits',
    '1140000': 'Money Market Instruments',
    '1710000': 'IC Receivable',
    '2710000': 'IC Payable'
}

print(f'Master data loaded: {len(vendors)} vendors, {len(customers)} customers, {len(company_codes)} entities')

Master data loaded: 10 vendors, 8 customers, 4 entities


## 3. Helper Functions
Utility functions for weighted random selection, date generation, document numbering, and realistic amount distributions.

In [3]:
def weighted_choice(choices_dict):
    """Pick a key from dict where values are probabilities."""
    keys = list(choices_dict.keys())
    weights = list(choices_dict.values())
    return np.random.choice(keys, p=weights)

def random_date(start, end):
    """Random date between start and end."""
    delta = (end - start).days
    random_days = np.random.randint(0, delta)
    return start + timedelta(days=random_days)

def generate_belnr(year, idx):
    """SAP document number — 10 digits, starts with 51 for FI docs."""
    return f'51{year % 100:02d}{idx:06d}'

def generate_amount(deal_type, doc_type):
    """
    Realistic amount ranges using lognormal distribution.
    - FX deals: large (hedging bulk vendor payments)
    - Vendor payments: large (hardware is expensive)
    - Customer payments: slightly larger (margin on top)
    - Other: varies widely
    """
    if deal_type in ('FX_SPOT', 'FX_FORWARD'):
        return round(np.random.lognormal(mean=12, sigma=1.2), 2)
    elif doc_type == 'KZ':
        return round(np.random.lognormal(mean=10, sigma=1.5), 2)
    elif doc_type == 'DZ':
        return round(np.random.lognormal(mean=10.2, sigma=1.3), 2)
    else:
        return round(np.random.lognormal(mean=9, sigma=1.8), 2)

print('Helper functions defined.')

Helper functions defined.


## 4. Main Data Generator
Generates 10,000 rows simulating a joined SAP ECC treasury extract.  
Each row represents a financial transaction line item — the kind of data a treasury analyst pulls every morning via SE16N or a custom ABAP report.

In [4]:
date_start = datetime(2024, 1, 1)
date_end = datetime(2024, 12, 31)

sap_users = ['TREAS01', 'TREAS02', 'TREAS03', 'FIADMIN', 'BATCH_F110', 'BATCH_FF7A']

rows = []

for i in range(NUM_ROWS):
    # --- Core identifiers ---
    bukrs = np.random.choice(list(company_codes.keys()), p=[0.40, 0.30, 0.20, 0.10])
    gjahr = 2024
    belnr = generate_belnr(gjahr, i)
    buzei = np.random.randint(1, 6)

    # --- Transaction type drives everything else ---
    blart = weighted_choice(doc_types)
    deal_type = weighted_choice(deal_types)
    deal_status = np.random.choice(
        ['OPEN', 'SETTLED', 'CANCELLED', 'PENDING'],
        p=[0.30, 0.50, 0.05, 0.15]
    )

    # --- Dates ---
    budat = random_date(date_start, date_end)
    bldat = budat - timedelta(days=np.random.randint(0, 5))
    cpudt = budat + timedelta(days=np.random.randint(0, 3))
    valut = budat + timedelta(days=np.random.randint(0, 4))

    # --- Counterparty logic: vendor OR customer, not both ---
    if blart in ('KZ', 'ZP'):
        lifnr = np.random.choice(list(vendors.keys()))
        kunnr = ''
        name1 = vendors[lifnr]['name']
        waers = vendors[lifnr]['currency']
    elif blart == 'DZ':
        lifnr = ''
        kunnr = np.random.choice(list(customers.keys()))
        name1 = customers[kunnr]['name']
        waers = customers[kunnr]['currency']
    else:
        if np.random.random() < 0.3:
            lifnr = np.random.choice(list(vendors.keys()))
            kunnr = ''
            name1 = vendors[lifnr]['name']
            waers = vendors[lifnr]['currency']
        else:
            lifnr = ''
            kunnr = ''
            name1 = company_codes[bukrs]
            waers = 'EUR'

    # --- Amounts ---
    wrbtr = generate_amount(deal_type, blart)
    rate = currencies_with_rates.get(waers, 1.0)
    rate_fluctuation = rate * np.random.uniform(0.97, 1.03)
    kursf = round(rate_fluctuation, 5)
    dmbtr = round(wrbtr * kursf, 2) if waers != 'EUR' else wrbtr

    # --- Posting key (debit/credit) ---
    if blart in ('KZ', 'ZP'):
        bschl = np.random.choice(['25', '50'])
    elif blart == 'DZ':
        bschl = np.random.choice(['15', '40'])
    else:
        bschl = np.random.choice(['40', '50'])

    # --- G/L Account ---
    gl_keys = list(gl_accounts.keys())
    if blart in ('KZ', 'ZP'):
        hkont = np.random.choice(['1600000', '1600001', '1100000', '1100001'])
    elif blart == 'DZ':
        hkont = np.random.choice(['1400000', '1400001', '1100000', '1100010'])
    elif deal_type in ('FX_SPOT', 'FX_FORWARD'):
        hkont = np.random.choice(['4800000', '4800001', '4800010', '4800011', '1100001'])
    else:
        hkont = np.random.choice(gl_keys)

    # --- Banking ---
    hbkid = np.random.choice(list(house_banks.keys()))
    hktid = np.random.choice(house_banks[hbkid]['accounts'])
    bankn = f'DE{np.random.randint(10, 99)}{np.random.randint(100000000, 999999999)}'

    # --- Payment method ---
    if waers == 'USD':
        rzawe = np.random.choice(['W', 'E'], p=[0.7, 0.3])
    else:
        rzawe = weighted_choice(payment_methods)

    # --- Treasury deal fields ---
    deal_id = f'FX{gjahr}{i:06d}' if deal_type.startswith('FX') else f'TR{gjahr}{i:06d}'
    maturity_date = (budat + timedelta(days=int(np.random.choice([30, 60, 90, 180, 365])))).strftime('%Y-%m-%d') \
        if deal_type in ('FX_FORWARD', 'DEPOSIT') else ''

    counterparty_bank = np.random.choice([
        'JPMorgan Chase', 'Citibank', 'HSBC', 'BNP Paribas',
        'Deutsche Bank Markets', 'Goldman Sachs', 'Barclays'
    ]) if deal_type.startswith('FX') else ''

    # --- Profit center ---
    prctr = np.random.choice(list(profit_centers.keys()), p=[0.40, 0.30, 0.15, 0.15])
    kostl = f'KS{bukrs}{prctr[-2:]}'

    # --- Item text ---
    text_templates = [
        f'Payment {name1}',
        f'FX {waers}/EUR spot deal',
        f'Transfer {bukrs} treasury',
        f'Invoice clearing {belnr}',
        f'Hedging {waers} exposure',
        f'Deposit maturity {maturity_date}',
        f'IC transfer to {bukrs}',
        f'Bank fee {hbkid}'
    ]
    sgtxt = np.random.choice(text_templates)

    # --- Assignment field ---
    zuession = f'PO-{np.random.randint(4500000, 4599999)}' if blart in ('KZ', 'ZP') \
        else f'INV-{np.random.randint(900000, 999999)}' if blart == 'DZ' \
        else ''

    # --- User and timestamp ---
    ernam = np.random.choice(sap_users)
    aedat = cpudt.strftime('%Y-%m-%d')

    # --- Build the row ---
    rows.append({
        'MANDT': '800',
        'BUKRS': bukrs,
        'GJAHR': gjahr,
        'BELNR': belnr,
        'BUZEI': buzei,
        'BUDAT': budat.strftime('%Y-%m-%d'),
        'BLDAT': bldat.strftime('%Y-%m-%d'),
        'VALUT': valut.strftime('%Y-%m-%d'),
        'CPUDT': cpudt.strftime('%Y-%m-%d'),
        'BLART': blart,
        'BSCHL': bschl,
        'HKONT': hkont,
        'SGTXT': sgtxt,
        'WRBTR': wrbtr,
        'DMBTR': dmbtr,
        'WAERS': waers,
        'KURSF': kursf,
        'LIFNR': lifnr,
        'KUNNR': kunnr,
        'NAME1': name1,
        'HBKID': hbkid,
        'HKTID': hktid,
        'BANKN': bankn,
        'RZAWE': rzawe,
        'DEAL_TYPE': deal_type,
        'DEAL_ID': deal_id,
        'DEAL_STATUS': deal_status,
        'MATURITY_DATE': maturity_date,
        'COUNTERPARTY_BANK': counterparty_bank,
        'PRCTR': prctr,
        'KOSTL': kostl,
        'ZUESSION': zuession,
        'ERNAM': ernam,
        'AEDAT': aedat
    })

df = pd.DataFrame(rows)
print(f'Generated {len(df)} rows, {len(df.columns)} columns')
df.head()

Generated 10000 rows, 34 columns


,MANDT,BUKRS,GJAHR,BELNR,BUZEI,BUDAT,BLDAT,VALUT,CPUDT,BLART,...,DEAL_TYPE,DEAL_ID,DEAL_STATUS,MATURITY_DATE,COUNTERPARTY_BANK,PRCTR,KOSTL,ZUESSION,ERNAM,AEDAT
0,800,1000,2024,5124000000,5,2024-05-01,2024-04-29,2024-05-03,2024-05-03,KZ,...,TRANSFER,TR2024000000,SETTLED,,,PC_HW,KS1000HW,PO-4585305,BATCH_FF7A,2024-05-03
1,800,1000,2024,5124000001,1,2024-07-06,2024-07-04,2024-07-09,2024-07-08,KZ,...,PAYMENT,TR2024000001,OPEN,,,PC_MS,KS1000MS,PO-4543001,TREAS01,2024-07-08
2,800,1000,2024,5124000002,2,2024-09-20,2024-09-18,2024-09-20,2024-09-21,SA,...,PAYMENT,TR2024000002,OPEN,,,PC_SV,KS1000SV,,BATCH_FF7A,2024-09-21
3,800,1000,2024,5124000003,1,2024-02-22,2024-02-21,2024-02-25,2024-02-22,AB,...,TRANSFER,TR2024000003,SETTLED,,,PC_HW,KS1000HW,,TREAS03,2024-02-22
4,800,1000,2024,5124000004,3,2024-02-10,2024-02-07,2024-02-10,2024-02-12,ZP,...,PAYMENT,TR2024000004,PENDING,,,PC_SW,KS1000SW,PO-4508110,BATCH_FF7A,2024-02-12


## 5. Dirty Data Injection
Real SAP exports are messy. This section injects realistic data quality issues:  
duplicates, missing values, mixed date formats, logical errors, currency mismatches, and text noise.

In [5]:
df_dirty = df.copy()
n = len(df_dirty)

# === 1. DUPLICATES (~3%) ===
dup_indices = np.random.choice(n, size=int(n * 0.03), replace=False)
duplicates = df_dirty.iloc[dup_indices].copy()
df_dirty = pd.concat([df_dirty, duplicates], ignore_index=True)
print(f'Added {len(duplicates)} duplicate rows. Total: {len(df_dirty)}')

# === 2. MISSING VALUES ===
valut_nulls = np.random.choice(n, size=int(n * 0.05), replace=False)
df_dirty.loc[valut_nulls, 'VALUT'] = np.nan

name_nulls = np.random.choice(n, size=int(n * 0.08), replace=False)
df_dirty.loc[name_nulls, 'NAME1'] = ''

fx_rows = df_dirty[df_dirty['WAERS'] != 'EUR'].index.tolist()
if fx_rows:
    kursf_nulls = np.random.choice(fx_rows, size=min(int(n * 0.02), len(fx_rows)), replace=False)
    df_dirty.loc[kursf_nulls, 'KURSF'] = np.random.choice([0, np.nan], size=len(kursf_nulls))
print(f'Injected nulls: VALUT={len(valut_nulls)}, NAME1={len(name_nulls)}, KURSF={len(kursf_nulls)}')

# === 3. DATE FORMAT INCONSISTENCIES ===
date_cols = ['BUDAT', 'BLDAT', 'CPUDT']
for col in date_cols:
    mess_indices = np.random.choice(n, size=int(n * 0.10), replace=False)
    for idx in mess_indices:
        val = df_dirty.at[idx, col]
        if pd.notna(val) and val != '':
            try:
                dt = pd.to_datetime(val)
                fmt = np.random.choice(['german', 'us'])
                if fmt == 'german':
                    df_dirty.at[idx, col] = dt.strftime('%d.%m.%Y')
                else:
                    df_dirty.at[idx, col] = dt.strftime('%m/%d/%Y')
            except:
                pass
print(f'Randomized date formats on {len(date_cols)} columns (~10% each)')

# === 4. FUTURE POSTING DATES ===
future_indices = np.random.choice(n, size=int(n * 0.005), replace=False)
for idx in future_indices:
    df_dirty.at[idx, 'BUDAT'] = (datetime(2025, 1, 1) + timedelta(days=np.random.randint(1, 180))).strftime('%Y-%m-%d')
print(f'Injected {len(future_indices)} future posting dates')

# === 5. VALUE DATE BEFORE POSTING DATE ===
valut_error_indices = np.random.choice(n, size=int(n * 0.01), replace=False)
for idx in valut_error_indices:
    budat_val = df_dirty.at[idx, 'BUDAT']
    if pd.notna(budat_val) and budat_val != '':
        try:
            dt = pd.to_datetime(budat_val)
            df_dirty.at[idx, 'VALUT'] = (dt - timedelta(days=np.random.randint(5, 30))).strftime('%Y-%m-%d')
        except:
            pass
print(f'Injected {len(valut_error_indices)} VALUT < BUDAT errors')

# === 6. CURRENCY ISSUES ===
curr_blank = np.random.choice(n, size=int(n * 0.01), replace=False)
df_dirty.loc[curr_blank, 'WAERS'] = ''

conv_error = np.random.choice(fx_rows, size=min(int(n * 0.008), len(fx_rows)), replace=False)
for idx in conv_error:
    df_dirty.at[idx, 'DMBTR'] = df_dirty.at[idx, 'WRBTR']
print(f'Injected currency issues: blank WAERS={len(curr_blank)}, bad conversions={len(conv_error)}')

# === 7. TEXT FIELD NOISE ===
text_mess_indices = np.random.choice(n, size=int(n * 0.15), replace=False)
for idx in text_mess_indices:
    val = df_dirty.at[idx, 'SGTXT']
    if pd.notna(val):
        mess_type = np.random.choice(['whitespace', 'case', 'special', 'empty'])
        if mess_type == 'whitespace':
            df_dirty.at[idx, 'SGTXT'] = '   ' + val + '  '
        elif mess_type == 'case':
            df_dirty.at[idx, 'SGTXT'] = val.lower() if np.random.random() > 0.5 else val.upper()
        elif mess_type == 'special':
            df_dirty.at[idx, 'SGTXT'] = val.replace('a', 'ä').replace('o', 'ö').replace('u', 'ü')
        else:
            df_dirty.at[idx, 'SGTXT'] = ''
print(f'Messed up {len(text_mess_indices)} SGTXT values')

# === 8. LOGICAL ERRORS ===
settled_rows = df_dirty[df_dirty['DEAL_STATUS'] == 'SETTLED'].index.tolist()
logic_err1 = np.random.choice(settled_rows, size=min(30, len(settled_rows)), replace=False)
for idx in logic_err1:
    df_dirty.at[idx, 'MATURITY_DATE'] = (datetime(2025, 6, 1) + timedelta(days=np.random.randint(1, 200))).strftime('%Y-%m-%d')

logic_err2 = np.random.choice(n, size=int(n * 0.005), replace=False)
for idx in logic_err2:
    df_dirty.at[idx, 'LIFNR'] = np.random.choice(list(vendors.keys()))
    df_dirty.at[idx, 'KUNNR'] = np.random.choice(list(customers.keys()))

fx_fwd = df_dirty[df_dirty['DEAL_TYPE'] == 'FX_FORWARD'].index.tolist()
logic_err3 = np.random.choice(fx_fwd, size=min(20, len(fx_fwd)), replace=False)
for idx in logic_err3:
    df_dirty.at[idx, 'MATURITY_DATE'] = ''

neg_indices = np.random.choice(n, size=int(n * 0.01), replace=False)
df_dirty.loc[neg_indices, 'WRBTR'] = -abs(df_dirty.loc[neg_indices, 'WRBTR'])

print(f'Logical errors: settled+future={len(logic_err1)}, dual counterparty={len(logic_err2)}, FX_FWD no maturity={len(logic_err3)}, negative amounts={len(neg_indices)}')

# === FINAL SHUFFLE ===
df_dirty = df_dirty.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n=== FINAL DIRTY DATASET ===')
print(f'Rows: {len(df_dirty)} (was {n}, added {len(df_dirty) - n} duplicates)')
print(f'Columns: {len(df_dirty.columns)}')
print(f'\nNull counts:\n{df_dirty.isnull().sum()[df_dirty.isnull().sum() > 0]}')

Added 300 duplicate rows. Total: 10300
Injected nulls: VALUT=500, NAME1=800, KURSF=200
Randomized date formats on 3 columns (~10% each)
Injected 50 future posting dates
Injected 100 VALUT < BUDAT errors
Injected currency issues: blank WAERS=100, bad conversions=80


C:\Users\bobur\AppData\Local\Temp\ipykernel_21524\2433620738.py:53: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dt = pd.to_datetime(budat_val)


Messed up 1500 SGTXT values
Logical errors: settled+future=30, dual counterparty=50, FX_FWD no maturity=20, negative amounts=100

=== FINAL DIRTY DATASET ===
Rows: 10300 (was 10000, added 300 duplicates)
Columns: 34

Null counts:
VALUT    495
KURSF    108
dtype: int64


## 6. Export Raw Dataset
Save the dirty data as both CSV and Excel — this is the "raw SAP export" artifact.

In [6]:
csv_path = os.path.join(OUTPUT_DIR, 'sap_ecc_treasury_export_raw.csv')
xlsx_path = os.path.join(OUTPUT_DIR, 'sap_ecc_treasury_export_raw.xlsx')

df_dirty.to_csv(csv_path, index=False, encoding='utf-8-sig')
df_dirty.to_excel(xlsx_path, index=False, engine='openpyxl')

print(f'CSV saved: {csv_path}')
print(f'Excel saved: {xlsx_path}')
print(f'\nCSV size: {os.path.getsize(csv_path) / (1024*1024):.1f} MB')
print(f'Excel size: {os.path.getsize(xlsx_path) / (1024*1024):.1f} MB')

CSV saved: c:\Users\bobur\Downloads\sap_ecc_treasury_export_raw.csv
Excel saved: c:\Users\bobur\Downloads\sap_ecc_treasury_export_raw.xlsx

CSV size: 2.7 MB
Excel size: 1.9 MB
